## 0. Setup & sélection des colonnes

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

df_raw = pd.read_csv("../data/raw/fr-esr-parcoursup_2020.csv", sep=";", encoding="utf-8-sig")

cols_cluster = [
    "Statut de l\u2019établissement de la filière de formation (public, privé…)",
    "Établissement",
    "Région de l\u2019établissement",
    "Sélectivité",
    "Filière de formation très agrégée",
    "Capacité de l\u2019établissement par formation",
    "Rang du dernier appelé du groupe 1",
    "Effectif total des candidats ayant accepté la proposition de l\u2019établissement (admis)",
    "Dont effectif des admis boursiers néo bacheliers",
    "Dont effectif des admis néo bacheliers avec mention Assez Bien au bac",
    "Dont effectif des admis néo bacheliers avec mention Bien au bac",
    "Dont effectif des admis néo bacheliers avec mention Très Bien au bac",
    "% d\u2019admis néo bacheliers boursiers",
    "% d\u2019admis dont filles",
]

df_cluster = df_raw[cols_cluster].copy()
df_cluster.columns = ["statut_etab", "etablissement", "region", "selectivite", "filiere_agregee", "capacite", "ran_grp1", "acc_tot", "acc_brs", 
                      "acc_ab", "acc_b", "acc_tb", "pct_bours", "pct_f",]

print(df_cluster.shape)
df_cluster.head(3)

(12760, 14)


,statut_etab,etablissement,region,selectivite,filiere_agregee,capacite,ran_grp1,acc_tot,acc_brs,acc_ab,acc_b,acc_tb,pct_bours,pct_f
0,Public,IFSI Séraphine de Senlis - CH Les Murets,Ile-de-France,formation selective,IFSI,72.0,1940.0,72,9,11,6,0,36.00,86.11
1,Public,AgroSup Dijon Direction Enseignement à Distanc...,Auvergne-Rhône-Alpes,formation selective,BTS,30.0,1.0,2,0,0,0,0,NaN,0.00
2,Public,Université Clermont Auvergne - Antenne de Vich...,Auvergne-Rhône-Alpes,formation non selec,Licence,45.0,90.0,45,3,9,25,11,6.67,51.11


## 1. Feature engineering initial

In [4]:
df_cluster["pct_brs"] = 100 * df_cluster["acc_brs"] / df_cluster["acc_tot"].replace(0, np.nan)
df_cluster["pct_ab"]  = 100 * df_cluster["acc_ab"]  / df_cluster["acc_tot"].replace(0, np.nan)
df_cluster["pct_b"]   = 100 * df_cluster["acc_b"]   / df_cluster["acc_tot"].replace(0, np.nan)
df_cluster["pct_tb"]  = 100 * df_cluster["acc_tb"]  / df_cluster["acc_tot"].replace(0, np.nan)

df_cluster = df_cluster.drop(columns=["acc_brs", "acc_ab", "acc_b", "acc_tb"])

print(df_cluster.shape)
print(df_cluster.isna().mean().sort_values(ascending=False) * 100)


(12760, 14)
ran_grp1           1.833856
pct_bours          1.246082
pct_brs            0.509404
pct_f              0.509404
pct_ab             0.509404
pct_b              0.509404
pct_tb             0.509404
capacite           0.007837
filiere_agregee    0.000000
etablissement      0.000000
statut_etab        0.000000
selectivite        0.000000
region             0.000000
acc_tot            0.000000
dtype: float64


## 2. Mapping des variables catégorielles

In [6]:
df_cluster["selectivite_bin"] = df_cluster["selectivite"].map({"formation selective": 1, "formation non selec": 0})

statut_map = {
    "Public": "Public",
    "Privé sous contrat d\'association": "Privé",
    "Privé enseignement supérieur": "Privé",
    "Privé hors contrat": "Privé",
}

df_cluster["statut_simplifie"] = df_cluster["statut_etab"].map(statut_map)
outre_mer = ["La Réunion", "Martinique", "Guadeloupe", "Polynésie française", "Guyane", "Mayotte", "Etranger"]

def to_zone(region):
    if region == "Ile-de-France":
        return "Ile-de-France"
    elif region in outre_mer:
        return "Outre-mer / Etranger"
    else:
        return "Province"

df_cluster["zone"] = df_cluster["region"].apply(to_zone)

print(df_cluster["selectivite_bin"].value_counts())
print(df_cluster["statut_simplifie"].value_counts())
print(df_cluster["zone"].value_counts())

selectivite_bin
1    10035
0     2725
Name: count, dtype: int64
statut_simplifie
Public    10171
Privé      2589
Name: count, dtype: int64
zone
Province                9735
Ile-de-France           2429
Outre-mer / Etranger     596
Name: count, dtype: int64


## 3. Nettoyage — Valeurs manquantes

In [8]:
# Vérification 1 : ran_grp1 manquant -> hypothèse "formation non sélective" (infirmée, cf. trace)
mask_rang_na = df_cluster["ran_grp1"].isna()
print(f"ran_grp1 manquant : {mask_rang_na.sum()} lignes")
print(pd.crosstab(df_cluster["selectivite"], mask_rang_na))

ran_grp1 manquant : 234 lignes
ran_grp1             False  True 
selectivite                      
formation non selec   2720      5
formation selective   9806    229


In [9]:
# Vérification 2 : pct_bours manquant -> hypothèse "acc_tot == 0"
mask_bours_na = df_cluster["pct_bours"].isna()
print(f"pct_bours manquant : {mask_bours_na.sum()} lignes")
print("dont acc_tot == 0 :", ((mask_bours_na) & (df_cluster["acc_tot"] == 0)).sum())
print("dont acc_tot != 0 (cas à investiguer) :", ((mask_bours_na) & (df_cluster["acc_tot"] != 0)).sum())

pct_bours manquant : 159 lignes
dont acc_tot == 0 : 65
dont acc_tot != 0 (cas à investiguer) : 94


In [10]:
# Vérification 3 : les 94 cas "pct_bours NaN + acc_tot != 0" -> hypothèse "acc_neobac == 0"
mask_bours_na_only = mask_bours_na & (df_cluster["acc_tot"] != 0)
col_neobac = "Effectif des admis néo bacheliers"
acc_neobac_check = df_raw.loc[mask_bours_na_only.values, col_neobac]
print("Combien ont acc_neobac == 0 :", (acc_neobac_check == 0).sum(), "/", len(acc_neobac_check))

Combien ont acc_neobac == 0 : 94 / 94


In [11]:
# Vérification 4 : les 229 "sélectives sans rang" -> quelle filière ?
mask_selective_no_rang = mask_rang_na & (df_cluster["selectivite"] == "formation selective")
print(df_cluster.loc[mask_selective_no_rang, "filiere_agregee"].value_counts())
print()
print("acc_tot sur ces lignes :")
print(df_cluster.loc[mask_selective_no_rang, "acc_tot"].describe())

filiere_agregee
Autre formation    212
BTS                 15
Licence              2
Name: count, dtype: int64

acc_tot sur ces lignes :
count    229.000000
mean      12.419214
std        9.834988
min        0.000000
25%        4.000000
50%       10.000000
75%       20.000000
max       39.000000
Name: acc_tot, dtype: float64


In [12]:
mask_capa_na = df_cluster["capacite"].isna()
overview = pd.DataFrame({"ran_grp1_na": mask_rang_na, "pct_bours_na": mask_bours_na, "capacite_na": mask_capa_na, 
                         "acc_tot_zero": df_cluster["acc_tot"] == 0,})
print(overview.sum())
print("Union des lignes concernées par au moins un problème :", overview.any(axis=1).sum(), "/", len(df_cluster))

ran_grp1_na     234
pct_bours_na    159
capacite_na       1
acc_tot_zero     65
dtype: int64
Union des lignes concernées par au moins un problème : 365 / 12760


**Décision** (détail des trois causes en trace écrite) : suppression des lignes où `ran_grp1`,
`capacite` ou `pct_bours` (et par cascade `pct_brs/f/ab/b/tb`) sont manquants — 365 lignes (2.86%),
toutes rattachées à l'une des trois causes vérifiées ci-dessus (formation sans admis, sans néo-bachelier
admis, ou en concours commun groupé sans rang individuel).

In [14]:
n_before = len(df_cluster)
cols_to_check = ["ran_grp1", "capacite", "pct_bours", "pct_brs", "pct_f", "pct_ab", "pct_b", "pct_tb"]
df_cluster_clean = df_cluster.dropna(subset=cols_to_check).copy()
n_after = len(df_cluster_clean)
print(f"Lignes avant : {n_before} | après : {n_after} | supprimées : {n_before - n_after} ({100*(n_before-n_after)/n_before:.2f}%)")
print()
print("NaN restants :", df_cluster_clean[cols_to_check].isna().sum().sum())

Lignes avant : 12760 | après : 12395 | supprimées : 365 (2.86%)

NaN restants : 0


In [15]:
# Vérification du biais introduit par la suppression (répartition par filière avant/après)
before = df_cluster["filiere_agregee"].value_counts(normalize=True) * 100
after = df_cluster_clean["filiere_agregee"].value_counts(normalize=True) * 100
comparison = pd.DataFrame({"avant (%)": before, "après (%)": after}).round(2)
comparison["écart (pts)"] = (comparison["après (%)"] - comparison["avant (%)"]).round(2)
print(comparison.sort_values("écart (pts)", key=abs, ascending=False))

                   avant (%)  après (%)  écart (pts)
filiere_agregee                                     
Autre formation        12.37      10.71        -1.66
BTS                    40.54      41.56         1.02
Licence                20.20      20.61         0.41
Ecole d'Ingénieur       3.15       2.79        -0.36
CPGE                    6.64       6.83         0.19
DUT                     6.30       6.49         0.19
IFSI                    2.58       2.65         0.07
PASS                    1.78       1.83         0.05
Licence_Las             3.58       3.61         0.03
Ecole de Commerce       1.13       1.16         0.03
EFTS                    1.73       1.75         0.02


## 4. Nettoyage : Doublons

In [17]:
dup_full = df_cluster_clean.duplicated().sum()
print(f"Doublons complets : {dup_full}")

dup_key = df_cluster_clean.duplicated(subset=["etablissement", "filiere_agregee"], keep=False)
print(f"Lignes partageant (établissement + filière très agrégée) : {dup_key.sum()}")
print("-> non conclusif en soi : filiere_agregee est une catégorie large (11 valeurs) sous laquelle")
print("   un même établissement propose plusieurs formations distinctes (vérifié sur échantillon,")
print("   cf. trace écrite). Aucune suppression.")

Doublons complets : 0
Lignes partageant (établissement + filière très agrégée) : 10308
-> non conclusif en soi : filiere_agregee est une catégorie large (11 valeurs) sous laquelle
   un même établissement propose plusieurs formations distinctes (vérifié sur échantillon,
   cf. trace écrite). Aucune suppression.


## 5. Nettoyage : Outliers

In [19]:
num_cols_outliers = ["capacite", "ran_grp1", "acc_tot", "pct_bours", "pct_brs", "pct_f", "pct_ab", "pct_b", "pct_tb"]

def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

header = f"{'colonne':12s} | {'n_outliers':>10s} | {'% outliers':>10s}"
print(header)
for c in num_cols_outliers:
    s = df_cluster_clean[c]
    lo, hi = iqr_bounds(s)
    n_out = ((s < lo) | (s > hi)).sum()
    print(f"{c:12s} | {n_out:10d} | {100*n_out/len(s):9.1f}%")

colonne      | n_outliers | % outliers
capacite     |       1427 |      11.5%
ran_grp1     |       1623 |      13.1%
acc_tot      |       1426 |      11.5%
pct_bours    |        237 |       1.9%
pct_brs      |        351 |       2.8%
pct_f        |          0 |       0.0%
pct_ab       |         63 |       0.5%
pct_b        |        300 |       2.4%
pct_tb       |       1428 |      11.5%


In [20]:
# Zoom sur les valeurs extrêmes : erreurs de saisie ou cas réels ? (cf. trace écrite pour l'interprétation)
for c in num_cols_outliers:
    print(f"--- Top 3 : {c} ---")
    print(df_cluster_clean.nlargest(3, c)[["etablissement", "filiere_agregee", c]].to_string(index=False))
    print()

--- Top 3 : capacite ---
etablissement filiere_agregee  capacite
         CNED             BTS    3400.0
         CNED             BTS    3200.0
         CNED             BTS    2400.0

--- Top 3 : ran_grp1 ---
     etablissement   filiere_agregee  ran_grp1
     ENSIM Le Mans Ecole d'Ingénieur   13932.0
      ESIREM Dijon Ecole d'Ingénieur   13920.0
ISTY Mantes-Vélizy Ecole d'Ingénieur   13919.0

--- Top 3 : acc_tot ---
                 etablissement filiere_agregee  acc_tot
Université Toulouse 1 Capitole         Licence     1589
           Université de Paris            PASS     1526
     Université Grenoble Alpes            PASS     1458

--- Top 3 : pct_bours ---
                                                   etablissement filiere_agregee  pct_bours
                                       Université de La Rochelle Autre formation      100.0
                                               Lycée Jules Verne Autre formation      100.0
Campus de l'excellence sportive de Bretagne - Ant

In [21]:
# Les pct_* à 100% sont-ils portés par de très faibles effectifs (instabilité statistique) ?
for c in ["pct_bours", "pct_brs", "pct_ab", "pct_b", "pct_tb"]:
    mask_100 = df_cluster_clean[c] == 100.0
    med = df_cluster_clean.loc[mask_100, "acc_tot"].median()
    n_small = (df_cluster_clean.loc[mask_100, "acc_tot"] < 5).sum()
    print(f"{c:10s} | lignes à 100% : {mask_100.sum():4d} | acc_tot médian : {med:5.1f} | acc_tot<5 : {n_small}")

pct_bours  | lignes à 100% :   43 | acc_tot médian :   5.0 | acc_tot<5 : 19
pct_brs    | lignes à 100% :    8 | acc_tot médian :   1.5 | acc_tot<5 : 8
pct_ab     | lignes à 100% :   12 | acc_tot médian :   1.0 | acc_tot<5 : 11
pct_b      | lignes à 100% :   25 | acc_tot médian :   2.0 | acc_tot<5 : 25
pct_tb     | lignes à 100% :   36 | acc_tot médian :  15.5 | acc_tot<5 : 13


**Conclusion outliers** : aucune suppression les valeurs extrêmes correspondent à des cas réels
(CNED pour les capacités extrêmes, écoles d'ingénieurs en concours commun pour les rangs élevés, grandes
licences universitaires pour les volumes d'admis, CPGE prestigieuses pour les 100% de mention). Traitement
prévu par transformation logarithmique plutôt que retrait de lignes (section 6). Limite documentée : les
`pct_*` à 100% sont majoritairement portés par de très petits effectifs — à mentionner en interprétation
des résultats du clustering.

## 6. Transformation des variables asymétriques

In [24]:
df_cluster_clean["capacite_log"] = np.log1p(df_cluster_clean["capacite"])
df_cluster_clean["ran_grp1_log"] = np.log1p(df_cluster_clean["ran_grp1"])

before = df_cluster_clean[["capacite", "ran_grp1"]].skew()
after = df_cluster_clean[["capacite_log", "ran_grp1_log"]].skew()
print("Skewness avant :\n", before)
print("\nSkewness après :\n", after)

Skewness avant :
 capacite    11.101413
ran_grp1     6.461625
dtype: float64

Skewness après :
 capacite_log    0.731387
ran_grp1_log    0.300166
dtype: float64


## 7. Feature engineering : traitement de la colinéarité `capacite` / `acc_tot`

In [26]:
df_cluster_clean["taux_remplissage"] = df_cluster_clean["acc_tot"] / df_cluster_clean["capacite"]
print(df_cluster_clean["taux_remplissage"].describe(percentiles=[.9, .95, .99, .999]))

count    12395.000000
mean         0.899602
std          0.191144
min          0.026250
90%          1.042621
95%          1.096440
99%          1.222940
99.9%        1.698180
max          2.515625
Name: taux_remplissage, dtype: float64


**Vérification** : 81 formations dépassent un taux de remplissage de 130% (cas légitimes : concours
communs multi-voies type NEOMA, activation de listes complémentaires post-bac vérifiés individuellement,
cf. trace écrite). Ces valeurs ne sont pas des erreurs mais restent rares (99e percentile ≈ 1.22) : elles
sont **écrêtées (winsorisées)** au 99e percentile plutôt que supprimées, pour neutraliser leur effet de
levier disproportionné sur les distances euclidiennes du clustering tout en conservant le signal.

In [28]:
p99 = df_cluster_clean["taux_remplissage"].quantile(0.99)
n_clipped = (df_cluster_clean["taux_remplissage"] > p99).sum()
print(f"Seuil de winsorisation (P99) : {p99:.3f} | lignes écrêtées : {n_clipped} ({100*n_clipped/len(df_cluster_clean):.2f}%)")

df_cluster_clean["taux_remplissage_clipped"] = df_cluster_clean["taux_remplissage"].clip(upper=p99)
print(df_cluster_clean["taux_remplissage_clipped"].describe())

Seuil de winsorisation (P99) : 1.223 | lignes écrêtées : 124 (1.00%)
count    12395.000000
mean         0.897654
std          0.185500
min          0.026250
25%          0.850000
50%          0.958333
75%          1.000000
max          1.222940
Name: taux_remplissage_clipped, dtype: float64


In [29]:
# Vérification finale de la multicolinéarité sur le jeu de features retenu
num_features = ["capacite_log", "ran_grp1_log", "taux_remplissage_clipped", "pct_bours", "pct_f", "pct_ab", "pct_b", "pct_tb"]
corr_matrix = df_cluster_clean[num_features].corr(method="spearman")
print(corr_matrix.round(2))
high_corr = [(num_features[i], num_features[j], round(corr_matrix.iloc[i, j], 2)) for i in range(len(num_features)) 
             for j in range(i + 1, len(num_features)) if abs(corr_matrix.iloc[i, j]) > 0.8]
print("\nPaires fortement corrélées (|corr| > 0.8) :", high_corr if high_corr else "aucune ✓")

                          capacite_log  ran_grp1_log  taux_remplissage_clipped  pct_bours  pct_f  pct_ab  pct_b  pct_tb
capacite_log                      1.00          0.71                      0.05      -0.06   0.17   -0.08   0.03    0.20
ran_grp1_log                      0.71          1.00                      0.13      -0.16   0.21   -0.19   0.10    0.29
taux_remplissage_clipped          0.05          0.13                      1.00       0.05   0.14    0.02   0.15    0.15
pct_bours                        -0.06         -0.16                      0.05       1.00   0.08    0.13  -0.13   -0.23
pct_f                             0.17          0.21                      0.14       0.08   1.00   -0.19   0.02    0.14
pct_ab                           -0.08         -0.19                      0.02       0.13  -0.19    1.00   0.04   -0.27
pct_b                             0.03          0.10                      0.15      -0.13   0.02    0.04   1.00    0.42
pct_tb                            0.20  

## 8. Encodage des catégorielles finales

In [31]:
dummies_statut = pd.get_dummies(df_cluster_clean["statut_simplifie"], prefix="statut", drop_first=True)
dummies_zone = pd.get_dummies(df_cluster_clean["zone"], prefix="zone", drop_first=True)
dummies_filiere = pd.get_dummies(df_cluster_clean["filiere_agregee"], prefix="filiere", drop_first=True)

print("Dummies statut :", dummies_statut.columns.tolist())
print("Dummies zone   :", dummies_zone.columns.tolist())
print("Dummies filiere:", dummies_filiere.columns.tolist())


Dummies statut : ['statut_Public']
Dummies zone   : ['zone_Outre-mer / Etranger', 'zone_Province']
Dummies filiere: ['filiere_BTS', 'filiere_CPGE', 'filiere_DUT', 'filiere_EFTS', "filiere_Ecole d'Ingénieur", 'filiere_Ecole de Commerce', 'filiere_IFSI', 'filiere_Licence', 'filiere_Licence_Las', 'filiere_PASS']


## 9. Standardisation

In [33]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_array = scaler.fit_transform(df_cluster_clean[num_features])

df_scaled = pd.DataFrame(scaled_array, columns=[f"{c}_scaled" for c in num_features],index=df_cluster_clean.index)
df_scaled["selectivite_bin"] = df_cluster_clean["selectivite_bin"].values

print(df_scaled.describe().round(2))

       capacite_log_scaled  ran_grp1_log_scaled  taux_remplissage_clipped_scaled  pct_bours_scaled  pct_f_scaled  pct_ab_scaled  pct_b_scaled  pct_tb_scaled  \
count             12395.00             12395.00                         12395.00          12395.00      12395.00       12395.00      12395.00       12395.00   
mean                  0.00                -0.00                            -0.00              0.00          0.00           0.00         -0.00           0.00   
std                   1.00                 1.00                             1.00              1.00          1.00           1.00          1.00           1.00   
min                  -3.43                -2.61                            -4.70             -1.51         -1.71          -1.75         -1.28          -0.56   
25%                  -0.58                -0.71                            -0.26             -0.70         -0.82          -0.69         -0.72          -0.56   
50%                  -0.10              

## 10. Assemblage de la matrice finale `X`

In [35]:
X = pd.concat([df_scaled, dummies_statut, dummies_zone, dummies_filiere], axis=1)
id_cols = df_cluster_clean[["etablissement", "filiere_agregee", "zone", "statut_simplifie", "region"]]
print("Shape de X :", X.shape)
X.head()

Shape de X : (12395, 22)


,capacite_log_scaled,ran_grp1_log_scaled,taux_remplissage_clipped_scaled,pct_bours_scaled,pct_f_scaled,pct_ab_scaled,pct_b_scaled,pct_tb_scaled,selectivite_bin,statut_Public,zone_Outre-mer / Etranger,zone_Province,filiere_BTS,filiere_CPGE,filiere_DUT,filiere_EFTS,filiere_Ecole d'Ingénieur,filiere_Ecole de Commerce,filiere_IFSI,filiere_Licence,filiere_Licence_Las,filiere_PASS
0,0.760031,1.819869,0.551752,0.535491,1.202309,-0.733611,-0.716586,-0.561856,1,True,False,False,False,False,False,False,False,False,True,False,False,False
2,0.222484,-0.150399,0.551752,-1.134937,0.018728,-0.419273,2.495091,0.832394,0,True,False,True,False,False,False,False,False,False,False,True,False,False
3,0.030335,-0.143362,0.551752,1.073696,1.672021,0.176313,-0.209479,-0.261658,1,True,False,True,False,False,False,True,False,False,False,False,False,False
4,0.059804,0.053996,1.104679,-1.514812,1.672021,-0.357352,-0.650683,-0.561856,1,False,False,False,False,False,False,True,False,False,False,False,False,False
5,0.550997,1.817542,-0.077203,-0.342153,-1.199345,-0.369035,-0.898379,-0.561856,0,True,False,True,False,False,False,False,False,False,False,True,False,False


In [36]:
assert X.isna().sum().sum() == 0, "Des NaN subsistent dans X !"
assert len(X) == len(id_cols) == len(df_cluster_clean), "Désalignement des index !"
assert X.select_dtypes(include="object").shape[1] == 0, "Il reste des colonnes texte dans X !"

final_corr = X[[c for c in X.columns if c.endswith("_scaled")]].corr()
high_corr_final = [(final_corr.columns[i], final_corr.columns[j], round(final_corr.iloc[i, j], 2)) for i in range(len(final_corr.columns)) 
                   for j in range(i + 1, len(final_corr.columns)) if abs(final_corr.iloc[i, j]) > 0.8]
print("Paires fortement corrélées :", high_corr_final if high_corr_final else "aucune ✓")
print()
print(f"X définitive : {X.shape[0]} lignes, {X.shape[1]} colonnes.")
print(f"Max abs(valeur scaled) : {X[[c for c in X.columns if c.endswith('_scaled')]].abs().values.max():.2f}")


Paires fortement corrélées : aucune ✓

X définitive : 12395 lignes, 22 colonnes.
Max abs(valeur scaled) : 5.52


In [59]:
import os

output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)
X.to_csv(f"{output_dir}/X_cluster.csv", index=True)
id_cols.to_csv(f"{output_dir}/id_cols_cluster.csv", index=True)
df_cluster_clean.to_csv(f"{output_dir}/df_cluster_clean.csv", index=True)

print("Fichiers exportés :")
for f in ["X_cluster.csv", "id_cols_cluster.csv", "df_cluster_clean.csv"]:
    path = f"{output_dir}/{f}"
    print(f"  {path} ({os.path.getsize(path)/1024:.0f} Ko)")

Fichiers exportés :
  ../data/processed/X_cluster.csv (2922 Ko)
  ../data/processed/id_cols_cluster.csv (926 Ko)
  ../data/processed/df_cluster_clean.csv (3078 Ko)
